# Volume 9, Chapter 11: Compact Groups (Python companion)

Haar random unitary matrices and the rigidity of their eigenvalues, Weyl's character formula and Schur functions by computer, moments of the trace and longest increasing subsequences, and the statistics behind random quantum circuit benchmarks.

*Before each solution, predict what the output will look like.*

## Exercises

**P11.1 (★) Rigid eigenvalues.** Sample Haar random unitary matrices (QR decomposition of a complex Gaussian matrix, with the phases of the diagonal of $R$ corrected, Volume 8, Theorem 8.7). Check Proposition 11.7, $\mathbb E|\operatorname{tr}U^k|^2=\min(k,n)$, for $n=5$ and $k=1,\dots,8$, and compare with $n$ independent uniform points on the circle.

**P11.2 (★★) Weyl's character formula by computer.** (a) Compute the Schur functions $s_\lambda=a_{\lambda+\delta}/a_\delta$ for $n=3$ symbolically, and list the weights with their multiplicities for $\lambda=(2,1,0)$ and $(3,0,0)$ (Figure 11.1). (b) Check the dimension formula of Theorem 11.6 for all $\lambda$ with $\lambda_1\le4$, $n=3$. (c) Check by Monte Carlo on $U(3)$ that the characters $s_\lambda$ are orthonormal.

**P11.3 (★★) Moments of the trace and increasing subsequences.** (a) Count the permutations of $\{1,\dots,a\}$ with longest increasing subsequence $\le n$, for $a\le8$ and $n\le4$, and compare with Monte Carlo estimates of $\mathbb E|\operatorname{tr}U|^{2a}$ (Theorem 11.9). (b) Plot the distribution of $\operatorname{tr}U$ for $n=50$ against the standard complex Gaussian (Corollary 11.10).

**P11.4 (★★) CS application: random quantum circuits.** A quantum computer with $q$ qubits applies a unitary $U$ of size $N=2^q$ to the state $|0\rangle$ and measures, producing the output $x$ with probability $p(x)=|U_{x0}|^2$. Experiments testing "quantum advantage" (Google's Sycamore experiment of 2019) use circuits that behave like Haar random unitaries. (a) Show that $Np(x)$ is approximately exponentially distributed (the *Porter–Thomas* law) and that $\mathbb E[p(x)^2]=\frac2{N(N+1)}$. (b) The *linear cross-entropy benchmark* $F=N\,\mathbb E[p(x)]-1$, with $x$ drawn from the device, is about $1$ for a perfect device and $0$ for a device that outputs uniform noise. Simulate both.

## Solutions

### P11.1

In [ ]:
import numpy as np, math, itertools
rng = np.random.default_rng(11)

def haar_unitary(n, rng):
    Z = (rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))) / np.sqrt(2)
    Q, R = np.linalg.qr(Z)
    d = np.diag(R)
    return Q * (d / np.abs(d))            # multiply column j by the phase of R_jj

n, M = 5, 20000
eig = np.array([np.linalg.eigvals(haar_unitary(n, rng)) for _ in range(M)])
iid = np.exp(2j * np.pi * rng.random((M, n)))
print(" k   E|tr U^k|^2   min(k,n)   independent points")
for k in range(1, 9):
    print(f"{k:2}   {np.mean(np.abs((eig ** k).sum(1)) ** 2):9.3f}   {min(k, n):6}   {np.mean(np.abs((iid ** k).sum(1)) ** 2):9.3f}")

The estimates match $\min(k,5)$ within Monte Carlo error: for $k<n$ the eigenvalues are much more evenly spread than independent points (variance $k$ instead of $n$), and for $k\ge n$ the power $U^k$ has "forgotten" the repulsion and behaves like independent points. This saturation at $k=n$ is the random-matrix analogue of the form factor in Montgomery's pair correlation (Volume 8, Chapter 8).

### P11.2

In [ ]:
import sympy as sp
from collections import Counter
t = sp.symbols('t1:4')

def alternant(mu):
    return sp.Matrix(3, 3, lambda j, k: t[k] ** mu[j]).det()

def schur(lam):
    delta = (2, 1, 0)
    return sp.Poly(sp.cancel(alternant([l + d for l, d in zip(lam, delta)]) / alternant(delta)), *t)

for lam in [(1, 0, 0), (2, 1, 0), (3, 0, 0)]:
    s = schur(lam)
    weights = Counter({m: c for m, c in zip(s.monoms(), s.coeffs())})
    print(lam, " dim =", sum(weights.values()), " weights:", dict(weights))

def dim_formula(lam):
    n = len(lam); num = den = 1
    for j in range(n):
        for k in range(j + 1, n):
            num *= lam[j] - lam[k] + k - j; den *= k - j
    return num // den

ok = all(sum(schur(lam).coeffs()) == dim_formula(lam)
         for lam in [(a, b, c) for a in range(5) for b in range(a + 1) for c in range(b + 1)])
print("dimension formula = s_lambda(1,1,1) for all lambda with lambda_1 <= 4:", ok)

$V_{(2,1,0)}$ has the six permutations of $(2,1,0)$ once and $(1,1,1)$ twice, $8$ in all; $V_{(3,0,0)}$ has the ten weights $(a,b,c)$ with $a+b+c=3$, each once: the two diagrams of Figure 11.1. The product formula agrees with $s_\lambda(1,1,1)$, the number of monomials counted with multiplicity. Now orthonormality, numerically, through the eigenvalues of random matrices of $U(3)$:

In [ ]:
lams = [(0, 0, 0), (1, 0, 0), (1, 1, 0), (2, 0, 0), (2, 1, 0), (1, 0, -1)]
def schur_num(lam, z):
    delta = np.array([2, 1, 0]); mu = np.array(lam) + delta
    A = np.linalg.det(z[:, None, :] ** mu[None, :, None])            # alternant, batched
    D = np.linalg.det(z[:, None, :] ** delta[None, :, None])
    return A / D
z = np.array([np.linalg.eigvals(haar_unitary(3, rng)) for _ in range(40000)])
S = np.array([schur_num(l, z) for l in lams])
G = (S @ S.conj().T) / z.shape[0]
print(np.round(G.real, 2))

The Gram matrix of the six characters is the identity up to Monte Carlo error (about $\pm0.02$): Schur functions evaluated at the eigenvalues of a random unitary matrix are orthonormal, which is Weyl's integration formula and Theorem 11.6 at work. (The character with $\lambda=(1,0,-1)$ is the adjoint representation of Exercise 11.1.)

### P11.3

In [ ]:
import bisect
def lis(seq):
    tops = []
    for x in seq:
        k = bisect.bisect_left(tops, x)
        if k == len(tops): tops.append(x)
        else: tops[k] = x
    return len(tops)

counts = {a: Counter(lis(p) for p in itertools.permutations(range(a))) for a in range(1, 9)}
def count_le(a, n): return sum(c for l, c in counts[a].items() if l <= n)

M = 100000
for n in (1, 2, 3, 4):
    tr = np.array([np.trace(haar_unitary(n, rng)) for _ in range(M)])
    row = []
    for a in range(1, 7):
        row.append(f"{np.mean(np.abs(tr) ** (2 * a)):8.2f}/{count_le(a, n):<4}")
    print(f"n = {n}:  " + "  ".join(row))
print("a! for comparison:", [math.factorial(a) for a in range(1, 7)])
print("Catalan numbers:  ", [math.comb(2 * a, a) // (a + 1) for a in range(1, 7)])

Each entry is "Monte Carlo estimate / exact count". For $n=1$, $|\operatorname{tr}U|=1$ and only the decreasing permutation has no increasing pair; for $n=2$ the counts are the Catalan numbers $1,2,5,14,42,132$ (Exercise 11.5); and the moment equals $a!$ as long as $a\le n$. High moments are dominated by rare large values of the trace, so the Monte Carlo estimates of the last columns are noisier, but they agree with the counts. Now the distribution of the trace for $n=50$:

In [ ]:
import matplotlib.pyplot as plt
tr = np.array([np.trace(haar_unitary(50, rng)) for _ in range(20000)])
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(tr.real[:3000], tr.imag[:3000], s=2, alpha=.4); ax[0].set_aspect('equal'); ax[0].set_title("tr U, n = 50")
x = np.linspace(-3, 3, 200)
ax[1].hist(np.sqrt(2) * tr.real, bins=60, density=True, alpha=.6, label="sqrt(2) Re tr U")
ax[1].plot(x, np.exp(-x ** 2 / 2) / np.sqrt(2 * np.pi), 'k', label="N(0,1)"); ax[1].legend(); plt.show()
print("E|tr U|^2 =", np.mean(np.abs(tr) ** 2).round(3), "  E|tr U|^4 =", np.mean(np.abs(tr) ** 4).round(3), " (complex Gaussian: 1 and 2)")

The cloud of traces is a round Gaussian blob of radius about $1$, even though $U$ has $50$ eigenvalues on the unit circle: the eigenvalues are so evenly spread that their sum almost cancels.

### P11.4

In [ ]:
for q in (4, 8, 10):
    N = 2 ** q
    P = np.concatenate([np.abs(haar_unitary(N, rng)[:, 0]) ** 2 for _ in range(max(1, 20000 // N))])
    print(f"q = {q:2}, N = {N:5}: E[p^2] = {np.mean(P ** 2):.3e}, 2/(N(N+1)) = {2 / (N * (N + 1)):.3e};"
          f"  P(N p > 1) = {np.mean(N * P > 1):.3f}  (Porter-Thomas: e^-1 = {math.exp(-1):.3f})")

q = 10; N = 2 ** q
U = haar_unitary(N, rng); p = np.abs(U[:, 0]) ** 2
ideal = rng.choice(N, size=5000, p=p / p.sum())
noise = rng.integers(0, N, 5000)
for name, xs in [("perfect device", ideal), ("uniform noise", noise)]:
    print(f"{name:15}: linear XEB = {N * p[xs].mean() - 1:+.3f}")

**Why.** The first column of a Haar unitary matrix is a uniform unit vector of $\C^N$ (Volume 8, Chapter 8), the normalised version of a complex Gaussian vector $g$; so $p(x)=|g_x|^2/\sum_y|g_y|^2$, and $|g_x|^2$ is exponential with mean $1$ while the denominator is close to $N$: $Np(x)$ is close to $\mathrm{Exp}(1)$, the Porter–Thomas law. Exactly, $p(x)$ has the Beta$(1,N-1)$ law, with $\mathbb E[p^2]=\frac2{N(N+1)}$.

For a perfect device, $x$ is drawn with probability $p(x)$, so $\mathbb E[p(x)]=\sum_xp(x)^2\approx\frac2N$ and $F\approx1$; for uniform noise, $\mathbb E[p(x)]=\frac1N$ and $F=0$. In between, a device whose output is the ideal one with probability $\phi$ and noise otherwise has $F\approx\phi$: the benchmark measures the fidelity. It is the second moment of a Haar matrix entry, a Schur-orthogonality computation, that makes the number $1$ appear.